In [ ]:
import hashlib
import re
from dataclasses import dataclass, field
from datetime import datetime
from typing import List, Optional
import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter, MarkdownHeaderTextSplitter


# ---------- 1. Data model ----------

@dataclass
class Chunk:
    id: str
    text: str
    doc_id: str
    metadata: dict = field(default_factory=dict)
    embedding: Optional[List[float]] = None


# ---------- 2. Loader ----------

class MarkdownLoader:
    """Loads .md files from a directory into (doc_id, text, base_metadata) tuples."""
    def load(self, path: str) -> List[dict]:
        import os
        docs = []
        for fname in os.listdir(path):
            if not fname.endswith(".md"):
                continue
            full_path = os.path.join(path, fname)
            with open(full_path, "r", encoding="utf-8") as f:
                text = f.read()
            docs.append({
                "id": fname,
                "text": text,
                "source": full_path,
            })
        return docs


# ---------- 3. Splitter (markdown-aware, header-preserving) ----------

class MarkdownAwareSplitter:
    def __init__(self, chunk_size=512, chunk_overlap=64):
        self.header_splitter = MarkdownHeaderTextSplitter(
            headers_to_split_on=[("#", "h1"), ("##", "h2"), ("###", "h3")]
        )
        self.recursive_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def split(self, doc: dict) -> List[Chunk]:
        header_sections = self.header_splitter.split_text(doc["text"])
        chunks = []
        idx = 0
        for section in header_sections:
            sub_chunks = self.recursive_splitter.split_text(section.page_content)
            section_path = " > ".join(
                v for k, v in section.metadata.items() if v
            )
            for sub_text in sub_chunks:
                chunk_id = f"{doc['id']}::chunk_{idx}"
                chunks.append(Chunk(
                    id=chunk_id,
                    text=sub_text,
                    doc_id=doc["id"],
                    metadata={
                        "doc_id": doc["id"],
                        "chunk_index": idx,
                        "source": doc["source"],
                        "section_path": section_path or None,
                        "created_at": datetime.utcnow().isoformat(),
                    }
                ))
                idx += 1
        return chunks


# ---------- 4. Deduper (exact match via content hash) ----------

class ExactDeduper:
    def __init__(self):
        self.seen_hashes = set()

    def filter(self, chunks: List[Chunk]) -> List[Chunk]:
        unique = []
        for chunk in chunks:
            normalized = re.sub(r"\s+", " ", chunk.text.strip().lower())
            h = hashlib.sha256(normalized.encode()).hexdigest()
            if h not in self.seen_hashes:
                self.seen_hashes.add(h)
                chunk.metadata["content_hash"] = h
                unique.append(chunk)
        return unique


# ---------- 5. Embedder (batched, local model) ----------

class LocalEmbedder:
    def __init__(self, model_name="all-MiniLM-L6-v2", batch_size=64):
        self.model = SentenceTransformer(model_name)
        self.batch_size = batch_size

    def embed_batch(self, chunks: List[Chunk]) -> List[Chunk]:
        texts = [c.text for c in chunks]
        vectors = self.model.encode(
            texts,
            batch_size=self.batch_size,
            show_progress_bar=True,
            normalize_embeddings=True,   # so cosine sim == dot product
        )
        for chunk, vector in zip(chunks, vectors):
            chunk.embedding = vector.tolist()
        return chunks

    # --- swap-in for OpenAI, same interface ---
    # def embed_batch(self, chunks):
    #     from openai import OpenAI
    #     client = OpenAI()
    #     for batch in batched(chunks, 100):
    #         resp = client.embeddings.create(model="text-embedding-3-small",
    #                                          input=[c.text for c in batch])
    #         for c, item in zip(batch, resp.data):
    #             c.embedding = item.embedding
    #     return chunks


# ---------- 6. Store (Chroma) ----------

class ChromaStore:
    def __init__(self, path="./chroma_db", collection_name="docs"):
        self.client = chromadb.PersistentClient(path=path)
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"},
        )

    def upsert(self, chunks: List[Chunk]):
        self.collection.upsert(
            ids=[c.id for c in chunks],
            embeddings=[c.embedding for c in chunks],
            documents=[c.text for c in chunks],
            metadatas=[c.metadata for c in chunks],
        )

    def delete_by_doc_id(self, doc_id: str):
        self.collection.delete(where={"doc_id": doc_id})

    def query(self, query_embedding: List[float], n_results=5, where: dict = None):
        return self.collection.query(
            query_embeddings=[query_embedding],
            n_results=n_results,
            where=where,
        )


# ---------- 7. Orchestrator ----------

class IndexingPipeline:
    def __init__(self, loader, splitter, deduper, embedder, store):
        self.loader = loader
        self.splitter = splitter
        self.deduper = deduper
        self.embedder = embedder
        self.store = store

    def run(self, source_path: str):
        documents = self.loader.load(source_path)

        chunks: List[Chunk] = []
        for doc in documents:
            chunks.extend(self.splitter.split(doc))
        print(f"Split {len(documents)} docs into {len(chunks)} chunks")

        chunks = self.deduper.filter(chunks)
        print(f"{len(chunks)} chunks after dedup")

        chunks = self.embedder.embed_batch(chunks)
        self.store.upsert(chunks)
        print(f"Indexed {len(chunks)} chunks")
        return chunks


# ---------- Usage ----------

if __name__ == "__main__":
    pipeline = IndexingPipeline(
        loader=MarkdownLoader(),
        splitter=MarkdownAwareSplitter(chunk_size=512, chunk_overlap=64),
        deduper=ExactDeduper(),
        embedder=LocalEmbedder(),
        store=ChromaStore(),
    )
    pipeline.run("./Documents")   # directory of .md files

    # --- Querying it back ---
    embedder = LocalEmbedder()
    store = ChromaStore()

    query = "How does chunk overlap work?"
    query_vec = embedder.model.encode([query], normalize_embeddings=True)[0].tolist()

    results = store.query(query_vec, n_results=3)
    for text, meta, dist in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0]
    ):
        print(f"\n[score={1 - dist:.3f}] {meta.get('section_path')}")
        print(text[:200], "...")